# Module 12-13 — Graph Algorithms

This is the worked reference notebook: run live in lecture, fully solved.
The version students receive with TODOs in place of the solved parts is
`assignments/pds/a11-graph-algorithms/starter/graph_algorithms.py`.

## 1. Dijkstra's algorithm, with path reconstruction (Lecture 1)

In [1]:
import heapq
from collections import deque

def dijkstra(adj, start):
    dist = {start: 0}
    prev = {start: None}
    visited = set()
    pq = [(0, start)]
    while pq:
        d, node = heapq.heappop(pq)
        if node in visited:
            continue
        visited.add(node)
        for neighbor, weight in adj.get(node, []):
            if neighbor in visited:
                continue
            nd = d + weight
            if nd < dist.get(neighbor, float('inf')):
                dist[neighbor] = nd
                prev[neighbor] = node
                heapq.heappush(pq, (nd, neighbor))
    return dist, prev

def reconstruct_path(prev, target):
    path = []
    node = target
    while node is not None:
        path.append(node)
        node = prev[node]
    return path[::-1]

edges = {0:[(1,10),(2,1)], 1:[(0,10),(2,1)], 2:[(0,1),(1,1)]}
dist, prev = dijkstra(edges, 0)
assert dist == {0:0, 1:2, 2:1}
assert reconstruct_path(prev, 1) == [0, 2, 1]
print("Dijkstra's checks passed, matching Lecture 1's exact trace")

Dijkstra's checks passed, matching Lecture 1's exact trace


## 2. A* search (Lecture 2)

In [2]:
def manhattan(a, b):
    return abs(a[0]-b[0]) + abs(a[1]-b[1])

def grid_neighbors(node, walls, size=3):
    x, y = node
    result = []
    for dx, dy in [(1,0),(-1,0),(0,1),(0,-1)]:
        nx, ny = x+dx, y+dy
        if 0 <= nx < size and 0 <= ny < size and (nx,ny) not in walls:
            result.append(((nx,ny), 1))
    return result

def a_star(start, goal, walls, size=3):
    dist = {start: 0}
    visited = set()
    pq = [(manhattan(start, goal), start)]
    explored = set()
    while pq:
        p, node = heapq.heappop(pq)
        if node in visited:
            continue
        visited.add(node)
        explored.add(node)
        if node == goal:
            break
        for nb, w in grid_neighbors(node, walls, size):
            nd = dist[node] + w
            if nd < dist.get(nb, float('inf')):
                dist[nb] = nd
                heapq.heappush(pq, (nd + manhattan(nb, goal), nb))
    return dist.get(goal), len(explored)

def dijkstra_grid(start, goal, walls, size=3):
    dist = {start: 0}
    visited = set()
    pq = [(0, start)]
    explored = set()
    while pq:
        d, node = heapq.heappop(pq)
        if node in visited:
            continue
        visited.add(node)
        explored.add(node)
        if node == goal:
            break
        for nb, w in grid_neighbors(node, walls, size):
            nd = d + w
            if nd < dist.get(nb, float('inf')):
                dist[nb] = nd
                heapq.heappush(pq, (nd, nb))
    return dist.get(goal), len(explored)

walls = {(1,1)}
d_dist, d_explored = dijkstra_grid((0,0), (2,0), walls)
a_dist, a_explored = a_star((0,0), (2,0), walls)
assert d_dist == a_dist == 2
assert a_explored <= d_explored
assert d_explored == 5 and a_explored == 3   # matches Lecture 2's exact measured numbers
print("A* checks passed: both find distance 2; A* explores", a_explored, "nodes vs Dijkstra's", d_explored)

A* checks passed: both find distance 2; A* explores 3 nodes vs Dijkstra's 5


## 3. Bellman-Ford, with negative-weight resolution (Lecture 3)

In [3]:
def bellman_ford(edge_list, start, n):
    dist = {start: 0}
    prev = {start: None}
    for _ in range(n - 1):
        for u, v, weight in edge_list:
            if u in dist and dist[u] + weight < dist.get(v, float('inf')):
                dist[v] = dist[u] + weight
                prev[v] = u
    return dist, prev

def has_negative_cycle(edge_list, dist):
    for u, v, weight in edge_list:
        if u in dist and dist[u] + weight < dist.get(v, float('inf')):
            return True
    return False

neg_edges = [(0,1,1), (0,2,4), (2,1,-10)]
dist, prev = bellman_ford(neg_edges, 0, 3)
assert dist == {0:0, 1:-6, 2:4}   # matches Lecture 3's exact trace
assert reconstruct_path(prev, 1) == [0, 2, 1]
assert not has_negative_cycle(neg_edges, dist)

# Confirm Dijkstra's gives the WRONG answer on this same graph (finalizes node 1 too early)
wrong_edges = {0:[(1,1),(2,4)], 1:[], 2:[(1,-10)]}
wrong_dist, _ = dijkstra(wrong_edges, 0)
assert wrong_dist[1] == 1   # WRONG, reproducing Lecture 3's demonstrated failure

# Negative cycle detection
cycle_edges = [(0,1,1),(1,2,1),(2,0,-3)]
cycle_dist, _ = bellman_ford(cycle_edges, 0, 3)
assert has_negative_cycle(cycle_edges, cycle_dist) == True
print("Bellman-Ford checks passed: correct answer -6 where Dijkstra's wrongly reports 1")

Bellman-Ford checks passed: correct answer -6 where Dijkstra's wrongly reports 1


## 4. Prim's algorithm, verified against Kruskal's (Lecture 4 / Week 10)

In [4]:
def prim(adj, start, n):
    visited = {start}
    mst = []
    edges = [(w, start, nb) for nb, w in adj[start]]
    heapq.heapify(edges)
    total = 0
    while edges and len(visited) < n:
        w, u, v = heapq.heappop(edges)
        if v in visited:
            continue
        visited.add(v)
        mst.append((u, v, w))
        total += w
        for nb, ww in adj[v]:
            if nb not in visited:
                heapq.heappush(edges, (ww, v, nb))
    return mst, total

def kruskal(n, edge_list):
    edge_list = sorted(edge_list)
    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            x = parent[x]
        return x
    mst = []
    for weight, u, v in edge_list:
        if find(u) != find(v):
            mst.append((u, v, weight))
            parent[find(u)] = find(v)
    return mst

mst_adj = {0:[(1,1),(2,3)], 1:[(0,1),(2,2),(3,5)], 2:[(0,3),(1,2),(3,4)], 3:[(1,5),(2,4)]}
prim_mst, prim_total = prim(mst_adj, 0, 4)
assert prim_total == 7   # matches Week 10 Lecture 3's Kruskal's result exactly

kruskal_edges = [(1,0,1),(2,1,2),(3,0,2),(4,2,3),(5,1,3)]
kruskal_mst = kruskal(4, kruskal_edges)
assert sum(w for _,_,w in kruskal_mst) == 7
print("Prim's checks passed: total weight 7, matching Kruskal's result on the identical graph")

Prim's checks passed: total weight 7, matching Kruskal's result on the identical graph
